# Data Profiling and Quality Checks
Goal: Check raw freMTPL2 data quality and build a clean policy-level severity table for EDA.


1. Imports & config (paths, seed)
2. Data loading (freq + sev)
3. Schema & basic info (shape, dtypes, head, fix types)
4. Quality checks
   - 4.a Duplicates
   - 4.b Missing values
   - 4.c Value ranges & cardinality
   - 4.d Join integrity (freq ↔ sev)
   - 4.e Outlier flagging (no treatment yet)

5. Build policy-level severity table (aggregate, join, target, weight)
6. Findings & decisions
7. Save to data/processed/


# I. Imports and comnfig

In [ ]:
# Import libraries
import time

In [ ]:
# SETUP AND CONFIGUATION
from pathlib import Path


# Locate project root via a marker file
def find_project_root(marker: str = "pyproject.toml") -> Path:
    """Walk up from the current directory until the marker file is found."""
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / marker).exists():
            return path
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


# Define project root and data directories

# Project root directory
PROJECT_ROOT = find_project_root()

# Data directories
DATA_RAW = PROJECT_ROOT / "data" / "raw"  # Raw data directory

# II. Data Loading

In [ ]:
# Import freMTPL2 data from sklearn
from sklearn.datasets import fetch_openml

# Start timer
start_time = time.time()

# freMTPL2 from OpenML: policies (frequency) and individual claims (severity)
freq = fetch_openml(data_id=41214, as_frame=True, parser="auto").frame
sev = fetch_openml(data_id=41215, as_frame=True, parser="auto").frame

# End timer and print elapsed time
end_time = time.time()
print(f"Data fetched in {end_time - start_time:.2f} seconds")

# III. Schema & basic info (shape, dtypes, head, fix types)

In [ ]:
# Shape of the datasets
print(f"Frequency dataset shape: {freq.shape}")
print(f"Severity dataset shape: {sev.shape}")

In [ ]:
# Display the first few rows of the frequency dataset
print("Frequency dataset preview:")
freq.head()

In [ ]:
# Display the first few rows of the severity dataset
print("Severity dataset preview:")
sev.head()

In [ ]:
# Display information about the frequency dataset
print("Frequency dataset information:")
freq.info()

In [ ]:
# Display information about the severity dataset
print("Severity dataset information:")
sev.info()

In [ ]:
# Fix types
# Convert 'IDpol' column to integer type in the frequency dataset
freq["IDpol"] = freq["IDpol"].astype(int)

# Remove '' in freq['VehGas'] and convert to category type
freq["VehGas"] = freq["VehGas"].replace("", "Unknown").astype("category")

# Check dtypes after conversion
print("Frequency dataset dtypes after conversion:")
print(freq.dtypes)

print("\nSeverity dataset dtypes after conversion:")
print(sev.dtypes)

# 4. Quality checks
   - 4.a Duplicates
   - 4.b Missing values
   - 4.c Value ranges & cardinality
   - 4.d Join integrity (freq ↔ sev)
   - 4.e Outlier flagging (no treatment yet)

### 4.a. Duplicates

In [ ]:
# Check for full-row duplicates in the frequency dataset
freq_duplicates = freq.duplicated(keep=False).sum()
print(f"Number of duplicate rows in the frequency dataset: {freq_duplicates}")

# Check for duplicates in IDpol column in the frequency dataset
idpol_duplicates = freq["IDpol"].duplicated(keep=False).sum()
print(f"Number of duplicate IDpol values in the frequency dataset: {idpol_duplicates}")

# Check for full-rowduplicates in the severity dataset
sev_duplicates = sev.duplicated(keep=False).sum()
print(f"Number of duplicate rows in the severity dataset: {sev_duplicates}")

# Check for duplicates in IDpol column in the severity dataset
idpol_sev_duplicates = sev["IDpol"].duplicated(keep=False).sum()
print(f"Number of duplicate IDpol values in the severity dataset: {idpol_sev_duplicates}")

In [ ]:
# Investigate duplicates in the severity dataset
# Display the duplicate rows in the severity dataset
if sev_duplicates > 0:
    print("Duplicate rows in the severity dataset:")
    print(sev[sev.duplicated(keep=False)])

In [ ]:
# Group the severity dataset by 'IDpol' and
# count the number of claims per policyalong with the total claim amount
sev_grouped = (
    sev.groupby("IDpol")
    .agg(total_claims=("ClaimAmount", "count"), total_claim_amount=("ClaimAmount", "sum"))
    .reset_index()
)

# Display the grouped severity dataset
print("Grouped severity dataset (by IDpol):")
sev_grouped.head()

In [ ]:
# Check for full-row duplicates in the grouped severity dataset
grouped_duplicates = sev_grouped.duplicated(keep=False).sum()
print(f"Number of duplicate rows in the grouped severity dataset: {grouped_duplicates}")

# Check for duplicates in IDpol column in the grouped severity dataset
idpol_grouped_duplicates = sev_grouped["IDpol"].duplicated(keep=False).sum()
print(f"Number of duplicate IDpol values in the grouped severity data: {idpol_grouped_duplicates}")